# Orbit Wars — 2-Player Render

Loads the specified agent and runs a 2-player game.

Run with Papermill to override the agent or opponent:
```bash
papermill render_2player.ipynb out.ipynb -p agent_file agent_v19.py -p opponent random
```

In [ ]:
agent_file = "agent_v20.py"
opponent = "random"

In [ ]:
%%capture
!pip install --upgrade "kaggle-environments>=1.28.0" papermill

In [ ]:
import importlib.util
import os

def load_agent(path):
    if path in ("random", "do_nothing"):
        return path
    if not os.path.isabs(path):
        path = os.path.join(os.getcwd(), path)
    spec = importlib.util.spec_from_file_location("_agent", path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    if hasattr(mod, "agent"):
        return mod.agent
    for name in dir(mod):
        obj = getattr(mod, name)
        if callable(obj) and not name.startswith("_"):
            return obj
    raise ValueError(f"No callable agent found in {path}")

agent = load_agent(agent_file)
opp = load_agent(opponent)
print(f"Agent:    {agent_file}")
print(f"Opponent: {opponent}")

In [ ]:
from kaggle_environments import make

env = make("orbit_wars", debug=True)
env.run([agent, opp])

final = env.steps[-1]
print(f"Game finished in {len(env.steps)} steps\n")
labels = [agent_file, opponent]
for i, s in enumerate(final):
    print(f"Player {i} ({labels[i]}): reward={s.reward}, status={s.status}")

In [ ]:
env.render(mode="ipython", width=800, height=600)